# Ensemble A — 2020 Presidential Election (Biden vs. Trump)

This notebook runs the ReCom ensemble scored on the 2020 presidential results (`PRES20D` / `PRES20R`).
Load the cleaned shapefile from `data/cleaned/pa_cleaned_precincts/` before running.


In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from functools import partial

from gerrychain import Graph, Partition, Election, MarkovChain
from gerrychain.updaters import Tally, cut_edges
from gerrychain.proposals import recom
from gerrychain.accept import always_accept
from gerrychain.constraints import within_percent_of_ideal_population

## Step 1 — Load Cleaned Shapefile

In [ ]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
shp_path = os.path.join(BASE_DIR, "data", "cleaned", "pa_cleaned_precincts", "pa_cleaned_precincts.shp")

precincts = gpd.read_file(shp_path)
precincts["SENDIST"] = precincts["SENDIST"].astype(int)

print("Shape:", precincts.shape)
print("Columns:", list(precincts.columns))
print("CRS:", precincts.crs)
print("Districts:", sorted(precincts["SENDIST"].unique()))
precincts.plot(column="SENDIST", figsize=(10, 8), cmap="tab20", edgecolor="black", linewidth=0.1, legend=False)
plt.title("PA Precincts — 2022 Senate Districts")
plt.axis("off")
plt.tight_layout()
plt.show()

## Step 2 — Build Dual Graph and Initial Partition

In [ ]:
graph = Graph.from_geodataframe(precincts)
print("Nodes:", graph.number_of_nodes(), "| Edges:", graph.number_of_edges())

election = Election("PRES20", {"Dem": "PRES20D", "Rep": "PRES20R"})

initial_partition = Partition(
    graph,
    assignment="SENDIST",
    updaters={
        "cut_edges": cut_edges,
        "population": Tally("TOTPOP", alias="population"),
        "PRES20": election,
    },
)

pops = sorted(initial_partition["population"].values())
print(f"Number of districts: {len(initial_partition)}")
print(f"Total population: {sum(pops):,}")
print(f"Ideal district population: {sum(pops)/len(initial_partition):,.0f}")
print(f"Min district pop: {min(pops):,}  |  Max: {max(pops):,}")
print(f"Enacted Dem seats (2020): {initial_partition['PRES20'].seats('Dem')}")

## Step 3 — Configure and Run ReCom Ensemble

- **Population tolerance:** 5 % of ideal district size (enacted plan deviates up to ~4.3 %, so 2 % would reject the initial state)
- **Steps:** 50 000 / 100 000 (two chains for convergence comparison)
- **Proposal:** ReCom — random spanning tree, split on a population-balanced edge

In [ ]:
POP_TOLERANCE = 0.05  # 5% — enacted plan deviates up to ~4.3% from ideal
NUM_STEPS = 50000

ideal_pop = sum(initial_partition["population"].values()) / len(initial_partition)

proposal = partial(
    recom,
    pop_col="TOTPOP",
    pop_target=ideal_pop,
    epsilon=POP_TOLERANCE,
    node_repeats=2,
)

chain = MarkovChain(
    proposal=proposal,
    constraints=[within_percent_of_ideal_population(initial_partition, POP_TOLERANCE)],
    accept=always_accept,
    initial_state=initial_partition,
    total_steps=NUM_STEPS,
)

dem_seats_list = []
dem_vote_shares_list = []

for i, partition in enumerate(chain):
    dem_seats_list.append(partition["PRES20"].seats("Dem"))
    dem_vote_shares_list.append(list(partition["PRES20"].percents("Dem")))
    if (i + 1) % 100 == 0:
        print(f"  step {i+1}/{NUM_STEPS}")

dem_seats = np.array(dem_seats_list)
dem_vote_shares = np.array(dem_vote_shares_list)
dem_vote_shares_sorted = np.sort(dem_vote_shares, axis=1)

print(f"\nDone. Collected {len(dem_seats)} plans.")
print(f"Dem seat range: {dem_seats.min()} – {dem_seats.max()}  |  median: {np.median(dem_seats):.0f}")

In [ ]:
POP_TOLERANCE = 0.05
NUM_STEPS_2 = 100000

proposal2 = partial(
    recom,
    pop_col="TOTPOP",
    pop_target=ideal_pop,
    epsilon=POP_TOLERANCE,
    node_repeats=2,
)

chain2 = MarkovChain(
    proposal=proposal2,
    constraints=[within_percent_of_ideal_population(initial_partition, POP_TOLERANCE)],
    accept=always_accept,
    initial_state=initial_partition,
    total_steps=NUM_STEPS_2,
)

dem_seats_list_2 = []
dem_vote_shares_list_2 = []

for i, partition in enumerate(chain2):
    dem_seats_list_2.append(partition["PRES20"].seats("Dem"))
    dem_vote_shares_list_2.append(list(partition["PRES20"].percents("Dem")))
    if (i + 1) % 200 == 0:
        print(f"  step {i+1}/{NUM_STEPS_2}")

dem_seats_2 = np.array(dem_seats_list_2)
dem_vote_shares_2 = np.array(dem_vote_shares_list_2)
dem_vote_shares_sorted_2 = np.sort(dem_vote_shares_2, axis=1)

print(f"\nDone. Collected {len(dem_seats_2)} plans.")
print(f"Dem seat range: {dem_seats_2.min()} – {dem_seats_2.max()}  |  median: {np.median(dem_seats_2):.0f}")

## Step 4 — Plot Results

In [ ]:
enacted_seats = initial_partition["PRES20"].seats("Dem")
enacted_percents = sorted(initial_partition["PRES20"].percents("Dem"))

# Rebuild clean arrays from both chain lists (safe to re-run even if variables were overwritten)
s1 = np.array(dem_seats_list)
vs1 = np.sort(np.array(dem_vote_shares_list), axis=1)
s2 = np.array(dem_seats_list_2)
vs2 = np.sort(np.array(dem_vote_shares_list_2), axis=1)

chains = [
    (s1, vs1, "50 000 steps", "steelblue"),
    (s2, vs2, "100 000 steps", "darkorange"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, (seats, vs_sorted, label, color) in enumerate(chains):
    # --- row 0: seat-count histogram ---
    ax = axes[0, col]
    all_seats = np.concatenate([s1, s2])
    bins = np.arange(all_seats.min() - 0.5, all_seats.max() + 1.5, 1)
    ax.hist(seats, bins=bins, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(enacted_seats, color="firebrick", linewidth=2, linestyle="--",
               label=f"Enacted: {enacted_seats}")
    ax.axvline(np.median(seats), color="black", linewidth=1.5, linestyle=":",
               label=f"Median: {np.median(seats):.0f}")
    ax.set_xlabel("Democratic Seats Won (out of 50)", fontsize=10)
    ax.set_ylabel("Number of Plans", fontsize=10)
    ax.set_title(f"Seat Count Distribution — {label}", fontsize=11)
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # --- row 1: sorted vote-share box plot ---
    ax2 = axes[1, col]
    n_dist = vs_sorted.shape[1]
    ax2.boxplot(
        [vs_sorted[:, i] for i in range(n_dist)],
        positions=range(1, n_dist + 1),
        widths=0.6,
        patch_artist=True,
        boxprops=dict(facecolor=color, alpha=0.5),
        medianprops=dict(color="black", linewidth=1.2),
        whiskerprops=dict(linewidth=0.7),
        capprops=dict(linewidth=0.7),
        showfliers=False,
    )
    ax2.plot(range(1, len(enacted_percents) + 1), enacted_percents,
             "r--o", markersize=3, linewidth=1.5, label="Enacted plan")
    ax2.axhline(0.5, color="black", linewidth=0.8)
    ax2.set_xlabel("District Rank (sorted by Dem vote share)", fontsize=10)
    ax2.set_ylabel("Democratic Vote Share", fontsize=10)
    ax2.set_title(f"Per-District Vote Shares — {label}", fontsize=11)
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax2.legend(fontsize=9)

plt.suptitle("ReCom Ensemble Comparison — PA State Senate, 2020 Presidential Results",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Step 5 — Save Ensemble Results

In [ ]:
out_dir = os.path.join(BASE_DIR, "data", "ensemble_results")
os.makedirs(out_dir, exist_ok=True)

pd.DataFrame({"dem_seats": dem_seats_list}).to_csv(
    os.path.join(out_dir, "ensemble_A_50000_seats.csv"), index=False
)
pd.DataFrame(np.sort(np.array(dem_vote_shares_list), axis=1),
             columns=[f"dist_{i+1}" for i in range(50)]).to_csv(
    os.path.join(out_dir, "ensemble_A_50000_vote_shares.csv"), index=False
)

pd.DataFrame({"dem_seats": dem_seats_list_2}).to_csv(
    os.path.join(out_dir, "ensemble_A_100000_seats.csv"), index=False
)
pd.DataFrame(np.sort(np.array(dem_vote_shares_list_2), axis=1),
             columns=[f"dist_{i+1}" for i in range(50)]).to_csv(
    os.path.join(out_dir, "ensemble_A_100000_vote_shares.csv"), index=False
)

print("Saved results to", out_dir)